# Notebook 01 — RLM Basics with smolagents

This notebook introduces the core concepts of **Recursive Language Models (RLMs)** and shows how the implementation in `src/rlm_smolagent.py` works.

## What is an RLM?

> *Recursive Language Models are a task-agnostic inference paradigm for language models to handle near-infinite length contexts by enabling the LM to programmatically examine, decompose, and recursively call itself over its input.*
> — [Zhang et al., 2025](https://arxiv.org/abs/2512.24601)

Traditional LLM call:
```python
response = llm.completion(prompt)   # context window is a hard limit
```

RLM call:
```python
response = rlm.completion(task, context)  # context stays in Python env
```

## Key design principle: context lives in the Python REPL, not the prompt

The critical difference from a naive LLM call is **where the context lives**.

| Approach | Where is the context? |
|---|---|
| Naive | Embedded as a string inside the prompt (hits context-window limit) |
| RLM | Stored as a Python variable `rlm_context` inside the REPL environment |

Because the context is a Python variable, the model can:
1. Inspect it: `len(rlm_context)`, `rlm_context.split(...)`, etc.
2. Extract slices: `chunk = rlm_context[:len(rlm_context)//2]`
3. Pass those slices to child calls using two available tools.

## Two sub-call tools (mirroring the paper's `llm_query` / `rlm_query`)

| Tool | Behaviour | When to use |
|---|---|---|
| `llm_call(sub_task, context_slice)` | Direct, non-recursive LLM call | Leaf-level queries on chunks small enough to answer in one shot |
| `rlm_call(sub_task, context_slice)` | Recursive RLM sub-call; child gets its own REPL | Complex sub-tasks that may need further decomposition |

The model decides freely in Python how to orchestrate:

```python
# Example A — paragraph-by-paragraph with direct llm_call (fast, no sub-REPL)
paragraphs = [p for p in rlm_context.split("\n\n") if p.strip()]
summaries = [llm_call(f"Summarise paragraph {i+1}", p)
             for i, p in enumerate(paragraphs)]
final_answer("\n".join(summaries))

# Example B — recursive binary split with rlm_call (deep decomposition)
mid   = len(rlm_context) // 2
left  = rlm_call("Summarise first half",  rlm_context[:mid])
right = rlm_call("Summarise second half", rlm_context[mid:])
final_answer(left + " " + right)

# Wrong — never embed the full context in a sub-call string:
rlm_call(f"Summarise: {rlm_context}")
```

## Architecture

```
 rlm.completion(task, context)
       │
       ├── rlm_context = context  (injected into REPL state, NOT the prompt)
       │
  CodeAgent REPL  ←──── model sees: task description + rlm_context variable
       │
       ├── llm_call("sub-task", slice)   →  direct LLM call (fast leaf call)
       ├── rlm_call("sub-task", slice)   →  child RLMAgent (depth+1, own REPL)
       │                                          │
       │                                 … until max_depth
       │                                 (plain LLM call with small slice)
       └── final_answer(aggregated_result)
```

## LLM backend

The container is configured to reach the **host llama.cpp server** via the `LLAMA_BASE_URL` environment variable (default: `http://host-gateway:8080/v1`).  The llama.cpp server exposes an OpenAI-compatible REST API, so the same `openai` Python client is used throughout.


## 1. Configuration

In [1]:
import os
import sys

# Make the src/ module importable from the notebook
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

# Connection settings — read from environment variables set by docker-compose,
# with sensible defaults for manual runs.
LLAMA_BASE_URL = os.environ.get('LLAMA_BASE_URL', 'http://host-gateway:1337')
LLAMA_MODEL    = os.environ.get('LLAMA_MODEL',    'Qwen3.5-35B-A3B-UD-IQ4_NL.gguf')
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', 'not-needed')

print(f'Backend URL : {LLAMA_BASE_URL}')
print(f'Model name  : {LLAMA_MODEL}')

Backend URL : http://host.docker.internal:1337
Model name  : Qwen3.5-35B-A3B-UD-IQ4_NL.gguf


## 2. Verify the llama.cpp server is reachable

In [2]:
from openai import OpenAI

client = OpenAI(base_url=LLAMA_BASE_URL, api_key=OPENAI_API_KEY)

try:
    models = client.models.list()
    print('✅ Server is reachable.  Available models:')
    for m in models.data:
        print(f'   • {m.id}')
except Exception as exc:
    print(f'❌ Could not reach the server: {exc}')
    print('   Make sure the llama.cpp server is running on the host and'
          ' LLAMA_BASE_URL points to it.')

✅ Server is reachable.  Available models:
   • Qwen3.5-35B-A3B-UD-IQ4_NL.gguf


## 3. Plain LLM completion (baseline)

In [8]:
resp = client.chat.completions.create(
    model=LLAMA_MODEL,
    messages=[{"role": "user", "content": "How many consecutive 'c' letters are in the word 'accede'?"}],
)
print('Plain LLM response:', resp.choices[0].message.content)

Plain LLM response: There are **2** consecutive 'c' letters in the word 'accede'.

Here is the breakdown: a-**cc**-e-e-d-e.


## 4. Import RLMAgent

In [9]:
from rlm_smolagent import RLMAgent

agent = RLMAgent(
    base_url=LLAMA_BASE_URL,
    model_name=LLAMA_MODEL,
    api_key=OPENAI_API_KEY,
    max_depth=2,     # allow up to 2 levels of recursion
    max_steps=8,     # max REPL cycles per depth level
    verbose=True,    # print agent reasoning
)
print('RLMAgent ready.')

RLMAgent ready.


## 5. Simple RLM completion

A short task with no separate context — demonstrates the agent REPL loop.
The `task` parameter is a short description; no context is needed here.

In [10]:
result = agent.completion(
    task="List the first 5 prime numbers, one per line."
)

print('\n=== Final Response ===')
print(result.response)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ You are an RLM (Recursive Language Model) agent at recursion depth 0/2.                                         │
│                                                                                                                 │
│ You run inside a Python REPL.  For any task that is too long or complex to                                      │
│ handle in a single pass, you should:                                                                            │
│   1. Split the input into manageable chunks.                                                                    │
│   2. Call `rlm_call(sub_prompt)` for each chunk.                                                                │
│   3. Aggregate the results in Python and produce the final answer.                                              │
│                                                                                                                 │
│ If the task is simple enough to answer directly, just do so.                                                    │
│                                                                                                                 │
│                                                                                                                 │
│ Task:                                                                                                           │
│ List the first 5 prime numbers, one per line.                                                                   │
│                                                                                                                 │
╰─ _TracingOpenAIServerModel - Qwen3.5-35B-A3B-UD-IQ4_NL.gguf ────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # The first 5 prime numbers are well-known: 2, 3, 5, 7, 11                                                       
  primes = [2, 3, 5, 7, 11]                                                                                        
                                                                                                                   
  # Print each prime number on a separate line                                                                     
  for prime in primes:                                                                                             
      print(prime)                                                                                                 
                                                                                                                   
  final_answer(primes)                                                                                             
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
2
3
5
7
11

Final answer: [2, 3, 5, 7, 11]

[Step 1: Duration 6.32 seconds| Input tokens: 2,353 | Output tokens: 192]


=== Final Response ===
[2, 3, 5, 7, 11]


## 6. Inspect the call tree (metadata)

In [11]:
import json

print(json.dumps(result.metadata, indent=2))

{
  "call_tree": {
    "prompt_preview": "List the first 5 prime numbers, one per line.",
    "depth": 0,
    "duration_s": 6.36,
    "response_preview": "[2, 3, 5, 7, 11]",
    "llm_requests": [],
    "children": []
  },
  "prompt_trace_enabled": false
}


## 6b. Using llm_call for direct leaf-level queries

`llm_call(sub_task, context_slice)` is a **direct, non-recursive** LLM call —
no child REPL is spawned.  It is the faster option when the context slice is
already small enough to answer in a single LLM call.

Below we split a list of sentences into two halves and summarise each half
directly with `llm_call`, then ask the root agent to combine the results.


In [ ]:
sentences = [
    "The cat sat on the mat.",
    "An RLM can process long texts by recursion.",
    "Python slices make decomposition easy.",
    "llm_call is a direct, non-recursive LLM call.",
    "rlm_call spawns a child agent with its own REPL.",
    "The parent aggregates child results into one answer.",
]
sentences_text = "\n".join(sentences)

# Build an agent that uses llm_call for leaf queries
leaf_agent = RLMAgent(
    base_url=LLAMA_BASE_URL,
    model_name=LLAMA_MODEL,
    api_key=OPENAI_API_KEY,
    max_depth=1,
    max_steps=8,
    verbose=True,
)

leaf_result = leaf_agent.completion(
    task=(
        "rlm_context is a newline-separated list of sentences.\n"
        "Split them into two halves using Python.\n"
        "Use llm_call() (NOT rlm_call) to get a one-sentence summary of each half.\n"
        "Combine both summaries into a final two-sentence answer."
    ),
    context=sentences_text,
)

print("=== llm_call demo response ===")
print(leaf_result.response)
print("\n=== Call tree ===")
import json
print(json.dumps(leaf_result.metadata, indent=2))


## 7. Recursive task — divide and conquer summation

We give the agent a list of 50 numbers **as a separate `context`** (not embedded
in the prompt string).  Inside the REPL the list is available as `rlm_context`.

A smart RLM should use Python to split the list and choose between the two tools:
- `llm_call(sub_task, half)` — if the half is small enough to sum directly
- `rlm_call(sub_task, half)` — if the half should be decomposed further


In [12]:
import random
random.seed(42)

numbers = [random.randint(1, 100) for _ in range(50)]
expected = sum(numbers)

# The task is a SHORT description — the numbers live in `context`.
# Inside the REPL they are available as the Python variable `rlm_context`.
result = agent.completion(
    task=(
        "rlm_context is a list of integers. "
        "Split it into two halves, use rlm_call() to sum each half "
        "by passing each half as the context_slice argument, "
        "then add the two partial sums together and return the total."
    ),
    context=numbers,   # stored as rlm_context Python variable, not embedded in prompt
)

print(f'Expected sum : {expected}')
print(f'RLM response : {result.response}')

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ You are an RLM (Recursive Language Model) agent at recursion depth 0/2.                                         │
│                                                                                                                 │
│ You run inside a Python REPL.  For any task that is too long or complex to                                      │
│ handle in a single pass, you should:                                                                            │
│   1. Split the input into manageable chunks.                                                                    │
│   2. Call `rlm_call(sub_prompt)` for each chunk.                                                                │
│   3. Aggregate the results in Python and produce the final answer.                                              │
│                                                                                                                 │
│ If the task is simple enough to answer directly, just do so.                                                    │
│                                                                                                                 │
│                                                                                                                 │
│ Task:                                                                                                           │
│ You have a list of numbers. Split it into two halves, use rlm_call() to sum each half recursively, then add the │
│ two partial sums together.                                                                                      │
│                                                                                                                 │
│ Numbers: [82, 15, 4, 95, 36, 32, 29, 18, 95, 14, 87, 95, 70, 12, 76, 55, 5, 4, 12, 28, 30, 65, 78, 4, 72, 26,   │
│ 92, 84, 90, 70, 54, 29, 58, 76, 36, 1, 98, 21, 90, 55, 44, 36, 20, 28, 98, 44, 14, 12, 49, 13\]                 │
│                                                                                                                 │
╰─ _TracingOpenAIServerModel - Qwen3.5-35B-A3B-UD-IQ4_NL.gguf ────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  numbers = [82, 15, 4, 95, 36, 32, 29, 18, 95, 14, 87, 95, 70, 12, 76, 55, 5, 4, 12, 28, 30, 65, 78, 4, 72, 26,   
  92, 84, 90, 70, 54, 29, 58, 76, 36, 1, 98, 21, 90, 55, 44, 36, 20, 28, 98, 44, 14, 12, 49, 13]                   
                                                                                                                   
  # Split into two halves                                                                                          
  mid = len(numbers) // 2                                                                                          
  first_half = numbers[:mid]                                                                                       
  second_half = numbers[mid:]                                                                                      
                                                                                                                   
  print(f"First half: {first_half}")                                                                               
  print(f"Second half: {second_half}")                                                                             
  print(f"First half length: {len(first_half)}")                                                                   
  print(f"Second half length: {len(second_half)}")                                                                 
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
First half: [82, 15, 4, 95, 36, 32, 29, 18, 95, 14, 87, 95, 70, 12, 76, 55, 5, 4, 12, 28, 30, 65, 78, 4, 72]
Second half: [26, 92, 84, 90, 70, 54, 29, 58, 76, 36, 1, 98, 21, 90, 55, 44, 36, 20, 28, 98, 44, 14, 12, 49, 13]
First half length: 25
Second half length: 25

Out: None

[Step 1: Duration 18.47 seconds| Input tokens: 2,571 | Output tokens: 1,167]

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 3 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  total_sum = 1113 + 1238                                                                                          
  final_answer(total_sum)                                                                                          
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: 2351

[Step 3: Duration 4.52 seconds| Input tokens: 9,784 | Output tokens: 1,414]

Expected sum : 2351
RLM response : 2351


In [13]:
# Inspect the recursive call tree
print(json.dumps(result.metadata, indent=2))

{
  "call_tree": {
    "prompt_preview": "You have a list of numbers. Split it into two halves, use rlm_call() to sum each half recursively, then add the two part\u2026",
    "depth": 0,
    "duration_s": 37.873,
    "response_preview": "2351",
    "llm_requests": [],
    "children": [
      {
        "prompt_preview": "Sum the following numbers: [82, 15, 4, 95, 36, 32, 29, 18, 95, 14, 87, 95, 70, 12, 76, 55, 5, 4, 12, 28, 30, 65, 78, 4, \u2026",
        "depth": 1,
        "duration_s": 5.311,
        "response_preview": "1113",
        "llm_requests": [],
        "children": []
      },
      {
        "prompt_preview": "Sum the following numbers: [26, 92, 84, 90, 70, 54, 29, 58, 76, 36, 1, 98, 21, 90, 55, 44, 36, 20, 28, 98, 44, 14, 12, 4\u2026",
        "depth": 1,
        "duration_s": 5.389,
        "response_preview": "1238",
        "llm_requests": [],
        "children": []
      }
    ]
  },
  "prompt_trace_enabled": false
}
